# KAN-CDSCO2004U  Machine Learning and Deep Learning

## Lab 15: Image Classification with CNNs — 📝 Exercises
**Estimated time: 1.5 hours**

### Learning Objectives
By the end of this exercise, you will be able to:
- Download and preprocess an **image dataset** using `ImageDataGenerator`
- Build a **Convolutional Neural Network (CNN)** with Keras
- **Train and evaluate** a CNN for binary image classification
- **Visualize** training vs validation accuracy and loss curves

**How to work through this notebook:**
- 🏃 **RUN** cells = Just execute the code to see the output
- ✏️ **TODO** cells = Write your own code or answer questions
- 📖 **READ** cells = Explanations to help you understand the concepts

## <center> Lecture-15 "Image Classification Using Convolutional Neural Networks"

📖 **READ**

# Classify Cats vs Dogs from Images Using a CNN

In this lab you will build an image classifier that decides whether a photo shows a **cat** or a **dog**.

**Task overview:**
- Build a CNN model consisting of **three convolutional blocks** (each followed by a MaxPooling layer), a **Flatten** layer, a **Dense(512, relu)** layer, and a final **Dense(1)** output layer.
- Use the **Adam** optimizer and **binary cross-entropy** loss.
- Train with `batch_size = 128`, `epochs = 15`, `IMG_HEIGHT = 150`, `IMG_WIDTH = 150`.

**Hints:**
- Use `tf.keras.Sequential` to stack layers.
- Use `tf.keras.preprocessing.image.ImageDataGenerator` to load and preprocess the images from disk.

---
# Part 1 — Data Preprocessing and Exploration
---

🏃 **RUN**

## Step 1: Import Libraries

Run this cell to import all the Python packages you will need throughout the exercise.

In [ ]:
# 🏃 RUN — Import packages
# Author: Luca Gudi (lgg.digi@cbs.dk)
# Src: TensorFlow.org
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Flatten, Dropout, MaxPooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator

🏃 **RUN**

## Step 2: Download and Extract the Dataset

The cell below downloads the **Cats vs Dogs** filtered dataset from Google's ML Education storage and extracts it automatically.

In [ ]:
# 🏃 RUN — Download the dataset
# The original Google zip link is broken, so we grab the images via
# tensorflow_datasets and save them into the folder layout the notebook expects.

import tensorflow_datasets as tfds
from PIL import Image

PATH = os.path.join(os.path.expanduser("~"), ".keras", "datasets", "cats_and_dogs_filtered")

for split in ("train", "validation"):
    for label in ("cats", "dogs"):
        os.makedirs(os.path.join(PATH, split, label), exist_ok=True)

if os.listdir(os.path.join(PATH, "train", "cats")):
    print("Dataset already exists at:", PATH)
else:
    print("Downloading cats_vs_dogs via tensorflow_datasets …")
    ds = tfds.load("cats_vs_dogs", split="train", as_supervised=True)

    LIMITS   = {"train": 1000, "validation": 500}          # images per class
    NAMES    = {0: "cats", 1: "dogs"}
    counters = {s: {0: 0, 1: 0} for s in LIMITS}

    for img, lbl in ds:
        lbl   = int(lbl)
        split = "train" if counters["train"][lbl] < LIMITS["train"] else "validation"
        if counters[split][lbl] >= LIMITS[split]:
            continue

        idx = counters[split][lbl]
        out = os.path.join(PATH, split, NAMES[lbl], f"{NAMES[lbl][:-1]}_{idx:04d}.jpg")
        Image.fromarray(img.numpy()).save(out)
        counters[split][lbl] += 1

        if all(counters[s][l] >= LIMITS[s] for s in LIMITS for l in (0, 1)):
            break

    for s in LIMITS:
        print(f"  {s:>10}: {counters[s][0]} cats, {counters[s][1]} dogs")

print("Dataset directory:", PATH)


In [ ]:
# 🏃 RUN — Alternative: download via the original zip (currently broken)
#
# _URL = 'https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip'
# path_to_zip = tf.keras.utils.get_file('cats_and_dogs.zip', origin=_URL, extract=True)
# PATH = os.path.join(os.path.dirname(path_to_zip), 'cats_and_dogs_filtered')

✏️ **TODO**

## Step 3: Set Up Data Directories

**Objective:** Define file paths for the training and validation sets.

**Instructions:**
1. Use `os.path.join(PATH, 'train')` to create `train_dir`.
2. Use `os.path.join(PATH, 'validation')` to create `validation_dir`.
3. Define paths for `train_cats_dir`, `train_dogs_dir`, `validation_cats_dir`, `validation_dogs_dir` using `os.path.join`.
4. Print all four paths to confirm they are correct.

In [ ]:
# ✏️ TODO — Define training and validation directory paths


In [ ]:
# ✏️ TODO — Define subdirectory paths for cats and dogs (train and validation)


✏️ **TODO**

## Step 4: Count Images and Set Hyperparameters

**Objective:** Understand how many images are in the dataset and define training parameters.

**Instructions:**
1. Use `len(os.listdir(...))` to count images in each subdirectory.
2. Compute `total_train` and `total_val`.
3. Print the counts.
4. Set `batch_size = 128`, `epochs = 15`, `IMG_HEIGHT = 150`, `IMG_WIDTH = 150`.

In [ ]:
# ✏️ TODO — Count training and validation images


In [ ]:
# ✏️ TODO — Set hyperparameters: batch_size, epochs, IMG_HEIGHT, IMG_WIDTH


✏️ **TODO**

## Step 5: Create Image Data Generators

**Objective:** Pre-process images for model training using `ImageDataGenerator`.

**Instructions:**
1. Create a `train_image_generator` using `ImageDataGenerator(rescale=1./255)`.
2. Create a `validation_image_generator` using `ImageDataGenerator(rescale=1./255)`.
3. Use `flow_from_directory` on the training generator:
   - Point to `train_dir`, set `target_size=(IMG_HEIGHT, IMG_WIDTH)`, `batch_size=batch_size`, `class_mode='binary'`, `shuffle=True`.
4. Use `flow_from_directory` on the validation generator:
   - Same parameters but no shuffling.

In [ ]:
# ✏️ TODO — Create ImageDataGenerators for training and validation


In [ ]:
# ✏️ TODO — Use flow_from_directory to load images into train_data_gen and val_data_gen


✏️ **TODO**

## Step 6: Visualize Sample Training Images

**Objective:** Verify the data loading by plotting sample images.

**Instructions:**
1. Use `next(train_data_gen)` to fetch one batch. Unpack it as `(sample_images, _)` to discard labels.
2. Define a `plotImages(images_arr)` function that:
   - Creates a subplot grid of 1 row × 5 columns using `plt.subplots(1, 5, figsize=(20, 20))`.
   - Flattens the axes, loops through the images, calls `ax.imshow(img)` and `ax.axis('off')`.
   - Calls `plt.tight_layout()` and `plt.show()`.
3. Call `plotImages(sample_images[:5])` to display 5 images.

In [ ]:
# ✏️ TODO — Fetch a batch of training images


In [ ]:
# ✏️ TODO — Define plotImages function and visualize 5 sample images


---
# Part 2 — Model Building
---

✏️ **TODO**

## Step 7: Define the CNN Architecture

**Objective:** Build a CNN for binary image classification.

**Instructions:**
1. Initialize a `Sequential` model.
2. Add a `Conv2D` layer: 16 filters, kernel size 3, padding `'same'`, activation `'relu'`, input shape `(IMG_HEIGHT, IMG_WIDTH, 3)`.
3. Add a `MaxPooling2D()` layer.
4. Add a `Conv2D` layer: 32 filters, kernel size 3, padding `'same'`, activation `'relu'`.
5. Add a `MaxPooling2D()` layer.
6. Add a `Conv2D` layer: 64 filters, kernel size 3, padding `'same'`, activation `'relu'`.
7. Add a `MaxPooling2D()` layer.
8. Add a `Flatten()` layer.
9. Add a `Dense(512, activation='relu')` layer.
10. Add a `Dense(1)` output layer (binary classification).

In [ ]:
# ✏️ TODO — Build the CNN model


✏️ **TODO**

## Step 8: Compile the Model

**Objective:** Configure the model for training.

**Instructions:**
1. Call `model.compile(...)` with:
   - `optimizer='adam'`
   - `loss=tf.keras.losses.BinaryCrossentropy(from_logits=True)`
   - `metrics=['accuracy']`
2. Call `model.summary()` to display the network structure.

In [ ]:
# ✏️ TODO — Compile the model


In [ ]:
# ✏️ TODO — Print model summary


---
# Part 3 — Model Training and Evaluation
---

✏️ **TODO**

## Step 9: Train the Model

**Objective:** Fit the CNN on the training data while monitoring validation performance.

**Instructions:**
1. Call `model.fit(...)` with:
   - `train_data_gen` as the first argument
   - `steps_per_epoch = total_train // batch_size`
   - `epochs = epochs`
   - `validation_data = val_data_gen`
   - `validation_steps = total_val // batch_size`
2. Store the result in a variable called `history`.

In [ ]:
# ✏️ TODO — Train the model


✏️ **TODO**

## Step 10: Visualize Training and Validation Metrics

**Objective:** Plot accuracy and loss curves to assess model performance.

**Instructions:**
1. Extract `acc`, `val_acc`, `loss`, `val_loss` from `history.history`.
2. Create `epochs_range = range(epochs)`.
3. Plot a figure with two subplots side-by-side:
   - **Left:** Training Accuracy vs Validation Accuracy (legend at `lower right`).
   - **Right:** Training Loss vs Validation Loss (legend at `upper right`).
4. Call `plt.show()`.

In [ ]:
# ✏️ TODO — Visualize training and validation accuracy and loss
